In [19]:
import pandas as pd
df = pd.read_csv('paragraph_question_30x.csv', encoding='utf-8-sig')

entries = []

for index, row in df.iterrows():
    query = row["Позитивный вопрос"]
    positive = row["Абзац"]

    entrie = {
        "query": query,
        "positive": positive,
    }
    entries.append(entrie)

In [20]:
for entrie in entries[:3]:
    print(f"""query: {entrie['query']}\npositive: {entrie['positive'][:100]}...
=========================================================================\n\n""")

query: Чем отличается установка IM101 в крейте расширения для резервированного и одиночного процессора?
positive: 2.3 Компоновка крейта расширения УСО Процессор с помощью модулей УСО принимает по физическим линиям ...


query: Чем отличается установка IM101 в крейте расширения для резервированного и одиночного процессора?
positive: 2.3 Компоновка крейта расширения УСО Процессор с помощью модулей УСО принимает по физическим линиям ...


query: Чем отличается установка IM101 в крейте расширения для резервированного и одиночного процессора?
positive: 2.3 Компоновка крейта расширения УСО Процессор с помощью модулей УСО принимает по физическим линиям ...




In [21]:
import random 
random.shuffle(entries)

# Делим на выборки
split_idx = int(0.8 * len(entries))

train_data = entries[:split_idx]
val_data = entries[split_idx:]

print(f"Train size: {len(train_data)}, Validation size: {len(val_data)}")

Train size: 24, Validation size: 6


In [22]:
# Преобразуем в DataFrame и сохраняем
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)

train_df.to_csv('train_data.csv', index=False, encoding='utf-8')
val_df.to_csv('val_data.csv', index=False, encoding='utf-8')

# Подготовка датасета в формате Hugging Face

In [23]:
import os
os.environ['WANDB_DISABLED'] = 'true'

In [24]:

from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import Dataset
import torch

In [25]:
def prepare_dataset(data):
    queries = []
    positives = []

    for item in data:
        queries.append(f"query: {item['query']}")
        positives.append(f"passage: {item['positive']}")

    return Dataset.from_dict({
        "anchor": queries,
        "positive": positives,
    })

train_dataset = prepare_dataset(train_data)
val_dataset = prepare_dataset(val_data)

In [26]:
# 3. Инициализация модели
model = SentenceTransformer("intfloat/multilingual-e5-small")

train_loss = losses.MultipleNegativesRankingLoss(model)

# 5. Подготовка Evaluator
def create_evaluator(dataset, train_data_1, test_data_1):
    import pandas as pd
    train_df = pd.DataFrame(train_data_1)
    test_df  = pd.DataFrame(test_data_1)

    # здесь вся выборка целиком
    whole_context = list(train_df.positive) + list(test_df.positive)

    corpus = {i: cont for i, cont in enumerate(whole_context)}

    test_df['idx'] = range(len(test_df))
    test_df = test_df.set_index('idx')

    queries = {}
    relevant_docs = {}

    for index, row in test_df.iterrows():
        q = row['query']
        queries[index] = q

        corrent_context_index = whole_context.index(row['positive'])
        relevant_docs[index] = [corrent_context_index]

    # print('queries', queries)

    # print('corpus', corpus)
    # print('relevant_docs', relevant_docs)

    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        show_progress_bar=True,
        accuracy_at_k=[3, 5]
    )

evaluator = create_evaluator(val_dataset, train_data, val_data)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [27]:
from transformers import TrainerCallback

class MetricsLogger:
    def __init__(self):
        self.logs = []

    def log(self, metrics):
        self.logs.append(metrics)

    def save(self, path):
        pd.DataFrame(self.logs).to_csv(path, index=False)

class CustomLoggingCallback(TrainerCallback):
    def __init__(self, metrics_logger):
        self.metrics_logger = metrics_logger

    def on_evaluate(self, args, state, control, **kwargs):
        if state.log_history:
            self.metrics_logger.log(state.log_history[-1])

# Инициализация
metrics_logger = MetricsLogger()
callback = CustomLoggingCallback(metrics_logger)

In [28]:
# 6. Настройка аргументов обучения
training_args = SentenceTransformerTrainingArguments(
    output_dir="./e5-retriever",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    warmup_steps=100,
    learning_rate=4e-5,
    fp16=True,
    eval_strategy="steps",
    eval_steps=2,
    save_steps=2,
    save_total_limit=3,
    load_best_model_at_end=True,
)

# 7. Инициализация Trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=train_loss,
    evaluator=evaluator,
)

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [29]:
trainer.add_callback(callback)
trainer.train()
metrics_logger.save("./training_metrics.csv")

Step,Training Loss,Validation Loss,Cosine Accuracy@3,Cosine Accuracy@5,Cosine Precision@1,Cosine Precision@3,Cosine Precision@5,Cosine Precision@10,Cosine Recall@1,Cosine Recall@3,Cosine Recall@5,Cosine Recall@10,Cosine Ndcg@10,Cosine Mrr@10,Cosine Map@100
2,No log,1.791759,1.000000,1.000000,1.000000,0.333333,0.200000,0.100000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,No log,1.791759,1.000000,1.000000,1.000000,0.333333,0.200000,0.100000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,No log,1.791759,1.000000,1.000000,1.000000,0.333333,0.200000,0.100000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  2.66it/s]


...

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  2.61it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [30]:
!pip install matplotlib
import matplotlib.pyplot as plt

In [32]:
model.save("/kaggle/working/e5_custom")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [33]:

results = evaluator(model)
print(evaluator.primary_metric)
print(results[evaluator.primary_metric])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  2.22it/s]

cosine_ndcg@10
1.0


In [34]:
!zip -r /kaggle/working/e5_custom.zip /kaggle/working/e5_custom

updating: kaggle/working/e5_custom/ (stored 0%)
updating: kaggle/working/e5_custom/sentence_bert_config.json (deflated 9%)
updating: kaggle/working/e5_custom/README.md (deflated 82%)
updating: kaggle/working/e5_custom/model.safetensors (deflated 36%)
updating: kaggle/working/e5_custom/1_Pooling/ (stored 0%)
updating: kaggle/working/e5_custom/1_Pooling/config.json (deflated 59%)
updating: kaggle/working/e5_custom/modules.json (deflated 62%)
updating: kaggle/working/e5_custom/config_sentence_transformers.json (deflated 40%)
updating: kaggle/working/e5_custom/tokenizer.json (deflated 76%)
updating: kaggle/working/e5_custom/2_Normalize/ (stored 0%)
updating: kaggle/working/e5_custom/tokenizer_config.json (deflated 50%)
updating: kaggle/working/e5_custom/config.json (deflated 52%)
